# Google Timeline Mobility Data

This notebook loads a two-day Google Timeline extract flattened into GeoParquet files for raw pings, semantic stops, semantic trips, and the display trajectory.

In [ ]:
from pathlib import Path

import contextily as cx
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

In [ ]:
data_dir = Path("data")

pings = gpd.read_parquet(data_dir / "my_timeline_rawSignals.parquet").sort_values("timestamp").set_index("timestamp")
stops = gpd.read_parquet(data_dir / "my_timeline_stops.parquet").sort_values("start_time").reset_index(drop=True)
trips = gpd.read_parquet(data_dir / "my_timeline_trips.parquet").sort_values("start_time").reset_index(drop=True)
timeline_path = gpd.read_parquet(data_dir / "my_timeline_timelinePath.parquet").sort_values("timestamp")

stops["stop_id"] = stops.index + 1

In [ ]:
print("Ping columns; timestamp is the index:")
print(pings.columns.tolist())
pings.head()

In [ ]:
print("Stop columns:")
print(stops.columns.tolist())
stops.head()

In [ ]:
stops[["stop_id", "start_time", "end_time", "duration_minutes", "place_semantic_type", "place_probability", "place_id"]]

In [ ]:
hourly_pings = pings.groupby("source").resample("h").size().unstack("source").fillna(0).astype(int)
time_zone = pings.index.tz

fig, ax = plt.subplots(figsize=(10, 3))
bottom = pd.Series(0, index=hourly_pings.index)

for source, color in zip(hourly_pings.columns, plt.get_cmap("tab10").colors):
    ax.bar(hourly_pings.index, hourly_pings[source], bottom=bottom, width=pd.Timedelta(minutes=50), color=color, label=source)
    bottom += hourly_pings[source]

ax.set_title("Raw position pings per hour")
ax.set_ylabel("pings")
ax.set_xlabel("")
ax.set_xlim(hourly_pings.index.min() - pd.Timedelta(minutes=30), hourly_pings.index.max() + pd.Timedelta(minutes=30))
ax.set_xticks(hourly_pings.index[::6])
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M", tz=time_zone))
ax.grid(axis="y", alpha=0.25)
ax.legend(title="source", ncol=3)
plt.tight_layout()

In [ ]:
pings_3857 = pings.to_crs(3857)
stops_3857 = stops.to_crs(3857)
trips_3857 = trips.to_crs(3857)
timeline_path_3857 = timeline_path.to_crs(3857)

mode_colors = {"IN_PASSENGER_VEHICLE": "#d7301f", "WALKING": "#4575b4"}
trip_colors = trips_3857["travel_mode"].map(mode_colors)

fig, ax = plt.subplots(figsize=(6.5, 10))

timeline_path_3857.plot(ax=ax, color="0.55", markersize=7, alpha=0.35, zorder=1)
pings_3857.plot(ax=ax, color="0.1", markersize=4, alpha=0.12, zorder=1)
trips_3857.plot(ax=ax, color=trip_colors, linewidth=4, alpha=0.9, zorder=2)
stops_3857.plot(ax=ax, color="gold", edgecolor="black", linewidth=0.9, markersize=stops_3857["duration_minutes"].clip(35, 240), zorder=3)

cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
ax.legend(
    handles=[
        Line2D([0], [0], color=mode_colors["IN_PASSENGER_VEHICLE"], lw=4, label="vehicle trip"),
        Line2D([0], [0], color=mode_colors["WALKING"], lw=4, label="walking trip"),
        Line2D([0], [0], marker="o", color="gold", markeredgecolor="black", lw=0, label="semantic stop"),
        Line2D([0], [0], marker="o", color="0.25", lw=0, markersize=4, label="raw ping"),
    ],
    loc="upper left",
)
ax.set_title("Google Timeline: trips, stops, and raw pings")
ax.axis("off")

plt.tight_layout()